# Study report: mask-time

The study asks which call sites of the kernel mask the interrupts, for
how long, and how the longest mask section grows with the load. It uses
the M4 method on the tree `ws-up-irqoff` (hooks
`CONFIG_ARM_IRQ_MASK_HOOKS` and `CONFIG_ARM_IRQ_MASK_HOOKS_ISR`). The
hooks record each mask section and each ISR body. The latency
histograms of these runs are not data (`LAT META m4=1`); the analysis
uses the M4 site tables only. All values are in cycles at 170 MHz. The
study `measurement-validity/mask-instrumentation` checks the method.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-mask-time@nucleo_g474re"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

In [ ]:
import plotly.graph_objects as go
from studylib import stats  # noqa: E402

FIXED = ["m4.mask.e14[z_arm_pendsv]", "m4.mask[arch_cpu_idle]", "m4.mask[z_impl_k_sem_give]", "m4.mask[z_impl_k_sem_take]"]
ADD = {ctx: f"m4.mask.{ctx}[z_add_timeout]" for ctx in ("e15", "e22", "t")}


def value(rep, metric, stat):
    """stats.value, or None if the data do not give the value."""
    try:
        return stats.value(rep, metric, stat)
    except stats.NoValue:
        return None


def point_table(experiment):
    """One row per point: the axes and the M4 values that the checks read."""
    if not nbfig.built(inst, experiment):
        return pd.DataFrame()
    axes = inst.rows(experiment).drop_duplicates("point")
    axes = axes[["point"] + [c for c in axes.columns if c.startswith(("axis.", "value."))]]
    out = []
    for _, tree, point, rep in inst.points(experiment):
        row = {"experiment": experiment, "tree": tree, "point": point,
               "longest mask section": value(rep, "m4.mask", "max(max_net)"),
               "place": value(rep, "m4.mask", "top_place"),
               "SysTick ISR body (max)": value(rep, "m4.isr.e15", "max(max)"),
               "longest section in a thread (max)": value(rep, "m4.mask.t", "max(max)"),
               "masked share": value(rep, "meta.m4_masked_share", "value")}
        for ctx, m in ADD.items():
            row[f"z_add_timeout {ctx} n"] = value(rep, m, "n")
            row[f"z_add_timeout {ctx} max_net"] = value(rep, m, "max(max_net)")
        for m in FIXED:
            row[m] = value(rep, m, "max(max_net)")
        out.append(row)
    df = pd.DataFrame(out).merge(axes, on="point", how="left")
    for a in ("timers", "sleepers"):
        if f"value.{a}" in df:
            df[a] = df[f"value.{a}"].astype(int)
        elif f"axis.{a}" in df:
            df[a] = df[f"axis.{a}"].str[1:].astype(int)
        else:
            df[a] = 0
    df["timeouts"] = df.timers + df.sleepers
    return df


sweep = point_table("wake-m4-sweep")
valid = point_table("wake-m4-validate")
thread = point_table("thread-m4")
if len(sweep):
    sweep["series"] = sweep["axis.slice"] + " s" + sweep.sleepers.astype(str)
    sweep = sweep.sort_values(["axis.slice", "sleepers", "timers"])

## The points

`wake-m4-sweep` (app `wake_latency`) has 24 points: 0, 16, 64 or 256
periodic load timers (`t*`) x 0, 8 or 64 sleeper threads (`s*`) x time
slicing on (`slice`, 20 ms) or off (`noslice`). `thread-m4` (app
`thread_latency`) has a semaphore or a mutex wake from a thread, with 0
or 64 load timers. `wake-m4-validate` is an import from the study
mask-instrumentation; its points `t64-*` and `t256-*` give values at 64
and 256 timers, with SysTick at the driver priority (`same`) or one
level lower (`lower`), and with the WHEEL backend (`t256-wheel`). The
table gives the values that the checks read. `max_net` is the max of a
site minus the floor of the calibration of the same log.

In [ ]:
COLS = ["experiment", "point", "timers", "sleepers", "timeouts", "longest mask section", "place", "SysTick ISR body (max)",
        "masked share"]
shown = [d[COLS] for d in (sweep, valid, thread) if len(d)]
pd.concat(shown, ignore_index=True) if shown else pd.DataFrame(columns=COLS)

## H1 a..c: the site of the longest mask section

H1 predicts that the DLIST insert walk in `z_add_timeout` is the
longest mask section. The lock is inlined in `z_add_timeout`, so all
callers share one site. The column `place` is the function of the mask
site with the largest `max_net` (and its inlined functions after
`via`). H1 b reads the points `t64-*` and `t256-*` of
`wake-m4-validate`; H1 c reads the points `t64` of `thread-m4`.

In [ ]:
top = pd.concat([d[["experiment", "point", "timeouts", "longest mask section", "place"]] for d in (sweep, valid, thread)
                 if len(d)], ignore_index=True) if any(len(d) for d in (sweep, valid, thread)) else pd.DataFrame(
    columns=["place"])
top["in z_add_timeout"] = top.place.fillna("").str.contains(r"\bz_add_timeout\b")
top

## H1 d: the longest mask section over the timeouts in the queue

H1 d expects that the max of `z_add_timeout` grows about linearly with
the timeouts in the queue: about 300 + 25 x (timers + sleepers), the
dashed line. The 25 cycles for each timeout come from the growth of the
ISR to thread p50; the design gives no basis for the 300 cycles. Each
line is one value of `slice` and of the sleepers. The markers of
`wake-m4-validate` and `thread-m4` are at their number of timers. A
person gives the verdict of H1 d.

In [ ]:
fig = go.Figure()
k = 0
if len(sweep):
    for s in dict.fromkeys(sweep.series):
        sub = sweep[sweep.series == s].sort_values("timeouts")
        fig.add_trace(go.Scatter(x=sub.timeouts, y=sub["longest mask section"], mode="lines+markers", name=f"wake-m4-sweep {s}",
            line={"color": nbfig.color(k), "width": 2}, marker={"size": 8, "symbol": nbfig.SYMBOLS[k % 6]},
            customdata=sub[["point", "place"]],
            hovertemplate="%{customdata[0]}: %{y} cycles<br>%{customdata[1]}<extra>wake-m4-sweep " + s + "</extra>"))
        k += 1
for name, d in (("wake-m4-validate", valid), ("thread-m4", thread)):
    if len(d):
        sub = d[d.timers > 0]
        fig.add_trace(go.Scatter(x=sub.timeouts, y=sub["longest mask section"], mode="markers", name=name,
                                 marker={"color": nbfig.color(k), "size": 12, "symbol": nbfig.SYMBOLS[k % 6],
                                         "line": {"color": "#fcfcfb", "width": 1}},
                                 customdata=sub[["point", "place"]],
                                 hovertemplate="%{customdata[0]}: %{y} cycles<br>%{customdata[1]}<extra>" + name
                                               + "</extra>"))
        k += 1
xmax = max([0] + [d.timeouts.max() for d in (sweep, valid, thread) if len(d)])
fig.add_trace(go.Scatter(x=[0, xmax], y=[300, 300 + 25 * xmax], mode="lines", name="300 + 25 x timeouts (H1 d)",
                         line={"color": "#52514c", "dash": "dash", "width": 1.5}, hoverinfo="skip"))
nbfig.style(nbfig.distinct(fig), "Longest mask section over the timeouts in the queue", "timeouts in the queue (timers + sleepers)",
            "max_net (cycles)").show()
if len(sweep):
    display(sweep.pivot_table(index="timers", columns=["axis.slice", "sleepers"], values="longest mask section",
                              aggfunc="first"))

## H1 e..l: the checks with a limit

Each marker is the value of one check at one point; the grey bar is the
limit of the check. `e`, `f` and `i` read the longest mask section of
`wake-m4-sweep` without sleepers, at 0, 64 and 256 timers. `g` and `h`
read 64 timers in `wake-m4-validate` and `thread-m4`. `j` reads the
SysTick entry of `z_add_timeout` at 256 timers. `k` reads the EXTI0
entry (the slice reset adds a timeout 20 ms ahead, so the walk stops
earlier). `l` reads the WHEEL backend, which has no insert walk.

In [ ]:
def rows_of(d, cid, sel, col, lo, hi):
    if not len(d):
        return []
    sub = d[sel(d)]
    return [{"check": f"H1 {cid}: {r.point}", "series": r.experiment, "value": r[col], "lo": lo, "hi": hi}
            for _, r in sub.iterrows()]


h1 = []
h1 += rows_of(sweep, "e", lambda d: (d.timers == 0) & (d.sleepers == 0), "longest mask section", 0, 999)
h1 += rows_of(sweep, "f", lambda d: (d.timers == 64) & (d.sleepers == 0), "longest mask section", 1800, 2600)
h1 += rows_of(valid, "g", lambda d: d.point.isin(["t64-same", "t64-lower"]), "longest mask section", 1800, 2600)
h1 += rows_of(thread, "h", lambda d: d.timers == 64, "longest mask section", 1800, 2600)
h1 += rows_of(sweep, "i", lambda d: (d.timers == 256) & (d.sleepers == 0), "longest mask section", 6300, 7500)
h1 += rows_of(valid, "j", lambda d: d.point.isin(["t256-same", "t256-lower"]), "z_add_timeout e15 max_net", 6300, 7500)
h1 += rows_of(sweep, "k", lambda d: (d.timers == 256) & (d.sleepers == 0) & (d["axis.slice"] == "slice"),
              "z_add_timeout e22 max_net", 3000, 3800)
h1 += rows_of(valid, "l", lambda d: d.point == "t256-wheel", "longest mask section", 0, 1499)
h1 = pd.DataFrame(h1)
if len(h1):
    nbfig.limits(h1, "H1: the longest mask section against the limit of each check", "max_net (cycles)").update_xaxes(
        tickangle=-40).show()
h1

## H2: time slicing and the contexts of z_add_timeout

`z_add_timeout` has one site for each context: SysTick (`e15`: the timer
restarts and the sleeper wakes), EXTI0 (`e22`: the slice reset in the
give) and thread (`t`: the slice reset at a switch, and `k_sleep()`). H2
predicts that without slicing the EXTI0 entry has no section (H2 a), and
the thread entry has no section without sleepers (H2 b). H2 c (manual)
expects that the SysTick entries stay. The menu selects the section
count `n` or the longest section `max_net`.

In [ ]:
if len(sweep) and not sweep[[f"z_add_timeout {c} n" for c in ADD]].fillna(0).to_numpy().any():
    print("No mask site of wake-m4-sweep has the function z_add_timeout (as its function or as an inlined function).")
if len(sweep):
    fig = go.Figure()
    for i, st in enumerate(("n", "max_net")):
        for k, ctx in enumerate(ADD):
            col = f"z_add_timeout {ctx} {st}"
            fig.add_trace(go.Bar(x=sweep.point, y=sweep[col], name=f"{ctx}", visible=(i == 0), marker_color=nbfig.color(k),
                                 hovertemplate="%{x}: %{y}<extra>z_add_timeout " + ctx + " " + st + "</extra>"))
    fig.update_layout(updatemenus=[{"buttons": [{"label": lab, "method": "update",
                                                 "args": [{"visible": [j // len(ADD) == i for j in range(2 * len(ADD))]},
                                                          {"yaxis.title.text": lab}]}
                                                for i, lab in enumerate(("sections (n)", "max_net (cycles)"))],
                                    "x": 1.0, "xanchor": "right", "y": 1.15, "yanchor": "top"}])
    nbfig.style(fig, "z_add_timeout by context, for each point of wake-m4-sweep", "point", "sections (n)",
                barmode="group").update_xaxes(tickangle=-45).show()
    display(sweep[["point", "axis.slice", "timers", "sleepers"] + [f"z_add_timeout {c} {s}" for c in ADD
                                                                   for s in ("n", "max_net")]])

## H3: the SysTick ISR body against the longest mask section

The SysTick ISR body contains the callbacks and the restarts of several
expiries in one tick. At the same priority, EXTI0 waits for the whole
body. H3 predicts that the SysTick ISR body is the largest blocking time
for EXTI0, and that its max grows faster with the timers than the
longest mask section (H3 a, manual). The figure shows the points of
`wake-m4-sweep` without sleepers (log y). The second figure gives the
checks with a limit (H3 b, d, e, f).

In [ ]:
if len(sweep):
    s0 = sweep[sweep.sleepers == 0]
    fig = go.Figure()
    k = 0
    for col in ("SysTick ISR body (max)", "longest mask section"):
        for sl in dict.fromkeys(s0["axis.slice"]):
            sub = s0[s0["axis.slice"] == sl].sort_values("timers")
            fig.add_trace(go.Scatter(x=sub.timers, y=sub[col], mode="lines+markers", name=f"{col}, {sl}",
                                     line={"color": nbfig.color(k), "width": 2, "dash": None if sl == "slice" else "dash"},
                                     marker={"size": 9, "symbol": nbfig.SYMBOLS[k % 6]}, customdata=sub.point,
                                     hovertemplate="%{customdata}: %{y} cycles<extra>" + f"{col}, {sl}" + "</extra>"))
            k += 1
    nbfig.style(nbfig.distinct(fig), "SysTick ISR body and longest mask section over the load timers (no sleepers)", "load timers",
                "cycles", yaxis_type="log").show()
    display(s0.pivot_table(index="timers", columns="axis.slice",
                           values=["SysTick ISR body (max)", "longest mask section"], aggfunc="first"))

In [ ]:
h3 = []
for cid, d, sel, lo, hi in (("b", valid, lambda d: d.point == "t64-same", 7500, 9500),
                            ("d", valid, lambda d: d.point == "t256-same", 33000, 42000),
                            ("e", sweep, lambda d: (d.timers == 256) & (d.sleepers == 0), 33000, 42000),
                            ("f", valid, lambda d: d.point == "t256-wheel", 6500, 9000)):
    if len(d):
        h3 += [{"check": f"H3 {cid}: {r.point}", "series": r.experiment, "value": r["SysTick ISR body (max)"],
                "lo": lo, "hi": hi} for _, r in d[sel(d)].iterrows()]
h3 = pd.DataFrame(h3)
if len(h3):
    nbfig.limits(h3, "H3: the SysTick ISR body max against the limit of each check", "max (cycles)").show()
if len(thread):
    print("H3 c (manual): the SysTick ISR body max in thread-m4 at 64 timers and in wake-m4-validate t64-same")
    display(pd.concat([thread[thread.timers == 64], valid[valid.point == "t64-same"] if len(valid) else valid])[
        ["experiment", "point", "SysTick ISR body (max)"]])
h3

## H4 a: the sites with a fixed cost

The middle part of PendSV, the wake window of `arch_cpu_idle`,
`k_sem_give` and `k_sem_take` have a fixed cost. H4 a expects that the
load does not change their longest section: the spread (max - min) /
min over all points of `wake-m4-sweep` is 0.10 or less for each site.

In [ ]:
if len(sweep):
    fig = go.Figure()
    for k, m in enumerate(FIXED):
        fig.add_trace(go.Scatter(x=sweep.point, y=sweep[m], mode="lines+markers", name=m,
                                 line={"color": nbfig.color(k), "width": 2},
                                 marker={"size": 9, "symbol": nbfig.SYMBOLS[k % 6]},
                                 hovertemplate="%{x}: %{y} cycles<extra>" + m + "</extra>"))
    nbfig.style(nbfig.distinct(fig), "The longest section of the sites with a fixed cost, for each point", "point",
                "max_net (cycles)").update_xaxes(tickangle=-45).show()
    spread = pd.DataFrame([{"check": f"H4 a: {m}", "series": "wake-m4-sweep", "min": sweep[m].min(), "max": sweep[m].max(),
                            "value": (sweep[m].max() - sweep[m].min()) / sweep[m].min() if sweep[m].min() else None,
                            "lo": 0, "hi": 0.10} for m in FIXED])
    nbfig.limits(spread, "H4 a: the spread of each site over the points", "(max - min) / min").update_layout(
        height=380).show()
    display(spread)

## H4 b, e, f: short sections without load

H4 b reads the point `cal-t0` of `wake-m4-validate` (no load): the
longest mask section and the longest ISR body. H4 e reads the longest
section in thread context of `thread-m4` with the semaphore and no
timers; H4 f the mutex lock and unlock sites of `thread-m4` with no
timers. The mutex does the priority inheritance under the lock.

In [ ]:
h4 = []
if len(valid):
    rep = next((rep for _, _, p, rep in inst.points("wake-m4-validate") if p == "cal-t0"), None)
    if rep is not None:
        for m in ("m4.mask", "m4.isr"):
            h4.append({"check": f"H4 b: cal-t0 {m}", "series": "wake-m4-validate", "value": value(rep, m, "max(max)"),
                       "lo": 0, "hi": 2000})
if len(thread):
    for _, _, p, rep in inst.points("thread-m4"):
        ax = thread[thread.point == p].iloc[0]
        if ax.timers != 0:
            continue
        if ax["axis.primitive"] == "sem":
            h4.append({"check": f"H4 e: {p} m4.mask.t", "series": "thread-m4", "value": value(rep, "m4.mask.t", "max(max)"),
                       "lo": 0, "hi": 1000})
        else:
            for m in ("m4.mask.t[z_impl_k_mutex_lock]", "m4.mask.t[z_impl_k_mutex_unlock]"):
                h4.append({"check": f"H4 f: {p} {m}", "series": "thread-m4", "value": value(rep, m, "max(max)"),
                           "lo": 0, "hi": 1499})
h4 = pd.DataFrame(h4)
if len(h4):
    nbfig.limits(h4, "H4 b, e, f: the longest section against the limit of each check", "max (cycles)").update_layout(
        height=420).show()
h4

## H4 d and the central result thread-sites: the sites in thread context

In `thread_latency` the giver is a thread, so its wake call and the
switch to the taker run in thread context. H4 d (manual) expects more
sites in thread context than in `wake_latency`: the wake call of the
giver, the switch (`arch_swap` ends the section of the lock in the wake
call), the slice abort and the slice add. The figure gives the longest
section of each thread site, for the points of `thread-m4` and for the
point `t0-s0-slice` of `wake-m4-sweep` (the colour is the point).

In [ ]:
parts = []
if nbfig.built(inst, "thread-m4"):
    parts.append(inst.sites("thread-m4").assign(experiment="thread-m4"))
if nbfig.built(inst, "wake-m4-sweep"):
    w = inst.sites("wake-m4-sweep")
    parts.append(w[w.point == "t0-s0-slice"].assign(experiment="wake-m4-sweep"))
ts = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["kind", "exc", "point", "function", "max_net"])
ts = ts[(ts.kind == "mask") & (ts.exc == "t") & (ts.n > 0)] if len(ts) else ts
if len(ts):
    ts["label"] = ts.experiment + " " + ts.point
    fig = go.Figure()
    for k, lab in enumerate(dict.fromkeys(ts.label)):
        sub = ts[ts.label == lab]
        fig.add_trace(go.Bar(x=sub.function, y=sub.max_net, name=lab, marker_color=nbfig.color(k),
                             customdata=sub[["place", "n"]],
                             hovertemplate="%{x} (%{customdata[0]}): %{y} cycles, n %{customdata[1]}<extra>" + lab
                                           + "</extra>"))
    nbfig.style(fig, "Mask sites in thread context", "function of the site", "max_net (cycles)",
                barmode="group").update_xaxes(tickangle=-40).show()
    display(ts.pivot_table(index="function", columns="label", values="max_net", aggfunc="max"))
    display(ts.groupby("label").function.nunique().rename("thread sites").to_frame())

## The masked share (central result masked-share)

The share of the M4 window with the interrupts masked
(`m4_masked_cyc` / `m4_window_cyc`), over the timeouts in the queue. The
grey bar at 0 timeouts is the limit of H4 c (no timers, no sleepers).
The comment of `wake-m4-sweep` said that the share grows with the load,
with no limit, so the share is a central result.

In [ ]:
if len(sweep):
    fig = go.Figure()
    base = sweep[(sweep.timers == 0) & (sweep.sleepers == 0)]
    if len(base):
        fig.add_trace(go.Bar(x=[0], y=[0.02], base=[0], width=[12], name="limit of H4 c",
                             marker={"color": "#c3c2b7", "opacity": 0.5}, hovertemplate="limit 0 .. 0.02<extra></extra>"))
    for k, s in enumerate(dict.fromkeys(sweep.series)):
        sub = sweep[sweep.series == s].sort_values("timeouts")
        fig.add_trace(go.Scatter(x=sub.timeouts, y=sub["masked share"], mode="lines+markers", name=s,
                                 line={"color": nbfig.color(k), "width": 2},
                                 marker={"size": 8, "symbol": nbfig.SYMBOLS[k % 6]}, customdata=sub.point,
                                 hovertemplate="%{customdata}: %{y:.2%}<extra>" + s + "</extra>"))
    nbfig.style(nbfig.distinct(fig), "Masked share of the M4 window", "timeouts in the queue (timers + sleepers)",
                "m4_masked_cyc / m4_window_cyc").update_yaxes(tickformat=".1%").show()
    display(sweep.pivot_table(index="timers", columns=["axis.slice", "sleepers"], values="masked share",
                              aggfunc="first"))